# M5 Forecasting — Baseline Models

Establishes the performance floor before any machine learning.
All LightGBM results are measured as improvement over these baselines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os

CACHE_FILE = "df_CA_cache.parquet"

if os.path.exists(CACHE_FILE):
    df_CA = pd.read_parquet(CACHE_FILE)
    print(f"Loaded from cache: {df_CA.shape}")
    print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
else:
    raise FileNotFoundError("Run 01_eda.ipynb first to generate df_CA_cache.parquet")

## 1. Compute Baseline Predictions

In [ ]:
print("Sorting by item_id, store_id, date...")
df_CA = df_CA.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)

grp = df_CA.groupby(["item_id", "store_id"], observed=True)["sales"]

# Baseline 1: Seasonal Naive — predict same day last week
df_CA["pred_naive"] = grp.shift(7).astype("float32")

# Baseline 2: Rolling Mean 28 — smoothed recent demand (shift(1) avoids leakage)
df_CA["pred_rolling28"] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
).astype("float32")

print("Baseline features computed.")
print(f"pred_naive    nulls: {df_CA['pred_naive'].isnull().sum():,}")
print(f"pred_rolling28 nulls: {df_CA['pred_rolling28'].isnull().sum():,}")

## 2. Train / Val Split (same window as model notebook)

In [ ]:
# Drop rows with no baseline prediction
df_base = df_CA.dropna(subset=["pred_naive", "pred_rolling28"]).reset_index(drop=True)

VAL_DAYS  = 28
max_date  = df_base["date"].max()
val_start = max_date - pd.Timedelta(days=VAL_DAYS - 1)

df_train = df_base[df_base["date"] <  val_start]
df_val   = df_base[df_base["date"] >= val_start].reset_index(drop=True)

print(f"Train: {df_train.shape[0]:>12,} rows  |  {df_train['date'].min().date()} → {df_train['date'].max().date()}")
print(f"Val:   {df_val.shape[0]:>12,} rows  |  {df_val['date'].min().date()} → {df_val['date'].max().date()}")

## 3. Evaluate — RMSSE per Series

In [ ]:
def compute_rmsse(df_train, df_val, pred_col):
    """
    RMSSE = sqrt(MSE_val / scale)
    scale = mean squared naive (lag-1) error on training data per series
    """
    # Scale: mean squared diff of consecutive training sales per series
    train_sorted = df_train.sort_values(["item_id", "store_id", "date"])
    scale = (
        train_sorted.groupby(["item_id", "store_id"], observed=True)["sales"]
        .apply(lambda x: np.mean(np.diff(x.values.astype(float)) ** 2))
        .reset_index()
        .rename(columns={"sales": "scale"})
    )
    scale["scale"] = scale["scale"].clip(lower=1e-8)

    # MSE per series on val
    mse = (
        df_val.groupby(["item_id", "store_id"], observed=True)
        .apply(
            lambda x: np.mean((x["sales"].astype(float) - x[pred_col].clip(lower=0)) ** 2),
            include_groups=False
        )
        .reset_index().rename(columns={0: "mse"})
    )

    rmsse_df = mse.merge(scale, on=["item_id", "store_id"])
    rmsse_df["rmsse"] = np.sqrt(rmsse_df["mse"] / rmsse_df["scale"])
    return rmsse_df


rmsse_naive   = compute_rmsse(df_train, df_val, "pred_naive")
rmsse_rolling = compute_rmsse(df_train, df_val, "pred_rolling28")

print("Seasonal Naive (lag_7):")
print(f"  Mean RMSSE:   {rmsse_naive['rmsse'].mean():.4f}")
print(f"  Median RMSSE: {rmsse_naive['rmsse'].median():.4f}")
print()
print("Rolling Mean 28:")
print(f"  Mean RMSSE:   {rmsse_rolling['rmsse'].mean():.4f}")
print(f"  Median RMSSE: {rmsse_rolling['rmsse'].median():.4f}")

In [ ]:
cat_lookup = df_val[["item_id", "store_id", "cat_id"]].drop_duplicates()

def rmsse_by_category(rmsse_df, label):
    return (
        rmsse_df.merge(cat_lookup, on=["item_id", "store_id"])
        .groupby("cat_id", observed=True)["rmsse"].median()
        .rename(label)
    )

cat_naive   = rmsse_by_category(rmsse_naive,   "Seasonal Naive")
cat_rolling = rmsse_by_category(rmsse_rolling, "Rolling Mean 28")

summary = pd.concat([cat_naive, cat_rolling], axis=1)
summary.loc["ALL"] = [rmsse_naive["rmsse"].median(), rmsse_rolling["rmsse"].median()]
print("Median RMSSE by Category:")
print(summary.round(4).to_string())

## 4. Visualise Baseline Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: RMSSE distribution comparison
axes[0].hist(rmsse_naive["rmsse"].clip(upper=5),   bins=60, color="steelblue", alpha=0.6, label="Seasonal Naive", density=True)
axes[0].hist(rmsse_rolling["rmsse"].clip(upper=5), bins=60, color="tomato",    alpha=0.6, label="Rolling Mean 28", density=True)
axes[0].set_title("RMSSE Distribution (clipped at 5)")
axes[0].set_xlabel("RMSSE")
axes[0].legend()

# Panel 2: Median RMSSE by category
cats = summary.index.astype(str)
x = np.arange(len(cats))
w = 0.35
axes[1].bar(x - w/2, summary["Seasonal Naive"],  w, color="steelblue", label="Seasonal Naive")
axes[1].bar(x + w/2, summary["Rolling Mean 28"], w, color="tomato",    label="Rolling Mean 28")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cats, rotation=15)
axes[1].set_title("Median RMSSE by Category")
axes[1].set_ylabel("Median RMSSE")
axes[1].legend()

# Panel 3: Sample item — actual vs both baselines
sample_item  = rmsse_naive.nsmallest(5, "rmsse").iloc[2]   # mid-range item
item_id, store_id = sample_item["item_id"], sample_item["store_id"]
sub = df_val[(df_val["item_id"] == item_id) & (df_val["store_id"] == store_id)]
axes[2].plot(sub["date"], sub["sales"],          label="Actual",          color="black",     linewidth=1.2)
axes[2].plot(sub["date"], sub["pred_naive"],      label="Seasonal Naive",  color="steelblue", linestyle="--")
axes[2].plot(sub["date"], sub["pred_rolling28"],  label="Rolling Mean 28", color="tomato",    linestyle="--")
axes[2].set_title(f"Sample: {item_id} @ {store_id}")
axes[2].set_xlabel("Date")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig("ca_baseline_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ca_baseline_evaluation.png")

## 5. Benchmark Summary

In [ ]:
benchmark = pd.DataFrame({
    "Model":         ["Seasonal Naive (lag_7)", "Rolling Mean 28"],
    "Mean RMSSE":    [rmsse_naive["rmsse"].mean(),   rmsse_rolling["rmsse"].mean()],
    "Median RMSSE":  [rmsse_naive["rmsse"].median(), rmsse_rolling["rmsse"].median()],
}).set_index("Model").round(4)

print("=" * 45)
print("BASELINE BENCHMARK — save these numbers.")
print("LightGBM must beat both to justify the complexity.")
print("=" * 45)
print(benchmark.to_string())
print()
print("Interpretation:")
print("  RMSSE < 1.0  → model beats naive forecast")
print("  RMSSE > 1.0  → model is WORSE than repeating last week's value")

# Save for use in results notebook
benchmark.to_csv("baseline_benchmark.csv")
print("\nSaved: baseline_benchmark.csv")